# Web Attack Detection — TensorFlow.js Model Creator (CIC-IDS-2017)

## For PWA Integration

This notebook will:

1. Load the CIC-IDS-2017 web attacks dataset
2. Engineer features from network flow statistics
3. Train a lightweight neural network classifier
4. Convert it to TensorFlow.js format
5. Export the model files for the PWA

**Runtime:** Select GPU runtime (Runtime → Change runtime type → GPU)





## Step 1: Install Required Libraries


In [12]:
!pip install -q tensorflow scikit-learn pandas numpy tensorflowjs

## Step 2: Import Libraries

In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import json, os, shutil

print("TensorFlow version:", tf.__version__)
print("✅ Libraries imported successfully")

TensorFlow version: 2.19.0
✅ Libraries imported successfully


## Step 3: Upload Dataset

Upload `Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv` to this Colab session.

Use the file explorer on the left (folder icon) → Upload button.

The dataset is from the CIC-IDS-2017 benchmark by the Canadian Institute for Cybersecurity and contains labeled network flows for benign traffic and web attacks (SQL injection, XSS, brute force).

## Step 4: Load and Inspect Dataset

In [14]:
# Load the CSV file
df = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
                 encoding='latin-1', low_memory=False)

# CIC-IDS has a leading-space issue in column names
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("\nColumn count:", len(df.columns))

# Find the label column
label_col = [c for c in df.columns if c.lower().strip() == 'label'][0]
print(f"\nLabel column: '{label_col}'")
print(f"\nClass distribution:")
print(df[label_col].value_counts())

Dataset shape: (170366, 79)

Column count: 79

Label column: 'Label'

Class distribution:
Label
BENIGN                          168186
Web Attack ï¿½ Brute Force        1507
Web Attack ï¿½ XSS                 652
Web Attack ï¿½ Sql Injection        21
Name: count, dtype: int64


## Step 5: Clean and Prepare Data

We:
- Remove rows with missing labels or infinite values
- Consolidate the three web attack sub-types (Brute Force, XSS, SQL Injection) into a single "Web Attack" class for binary classification
- Drop tiny classes that can't be trained reliably

In [15]:
# Remove rows with missing labels or infinite feature values
df = df.dropna(subset=[label_col])
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# Consolidate web attack sub-types into a single "Web Attack" class
df[label_col] = df[label_col].str.strip()
df[label_col] = df[label_col].apply(
    lambda x: 'Web Attack' if 'Web Attack' in x else x
)

# Drop classes with fewer than 100 samples
counts = df[label_col].value_counts()
keep = counts[counts >= 100].index
df = df[df[label_col].isin(keep)].copy()

print(f"✅ Cleaned dataset: {len(df):,} rows")
print(f"\nFinal class distribution:")
print(df[label_col].value_counts())

✅ Cleaned dataset: 170,231 rows

Final class distribution:
Label
BENIGN        168051
Web Attack      2180
Name: count, dtype: int64


## Step 6: Prepare Features

We use all 78 pre-computed network flow statistics from CIC-IDS-2017 as features (flow duration, packet counts, byte counts, flow rates, etc.). These are extracted from raw packet captures by the CICFlowMeter tool.

In [16]:
feature_cols = [c for c in df.columns if c != label_col]
X = df[feature_cols].values.astype(np.float32)

print(f"✅ Features prepared: {len(feature_cols)} columns")
print(f"Feature matrix shape: {X.shape}")

✅ Features prepared: 78 columns
Feature matrix shape: (170231, 78)


## Step 7: Encode Labels and Normalize Features

- Convert string labels (BENIGN, Web Attack) to integers (0, 1)
- Apply StandardScaler to normalize features to zero mean and unit variance

In [17]:
# Encode labels
le = LabelEncoder()
y = le.fit_transform(df[label_col])
class_names = list(le.classes_)
print(f"Classes: {class_names}")

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"✅ Features normalized (zero mean, unit variance)")

Classes: ['BENIGN', 'Web Attack']
✅ Features normalized (zero mean, unit variance)


## Step 8: Split Data into Training and Test Sets

80/20 split with stratification to preserve class balance in both sets.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")

# Compute class weights to handle class imbalance
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))
print(f"\nClass weights (for imbalance): {class_weights}")

Training samples: 136,184
Test samples: 34,047

Class weights (for imbalance): {0: np.float64(0.5064861648318952), 1: np.float64(39.043577981651374)}


## Step 9: Build the Neural Network

A small Multi-Layer Perceptron (MLP):

- Input: 78 features
- Dense(128) + Dropout(0.3)
- Dense(64) + Dropout(0.2)
- Dense(2, softmax) → output probabilities

Dropout prevents overfitting. We use Adam optimizer with sparse categorical crossentropy loss, standard for multi-class classification with integer labels.

In [19]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_scaled.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │        10,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,498 (72.26 KB)

 Trainable params: 18,498 (72.26 KB)

 Non-trainable params: 0 (0.00 B)

## Step 10: Train the Model

15 epochs, batch size 256, with class weights to handle the imbalance (benign traffic vastly outnumbers attacks).

In [20]:
print("Training...")
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    class_weight=class_weights,
    verbose=1
)
print("\n✅ Training complete!")

Training...
Epoch 1/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9657 - loss: 0.1051 - val_accuracy: 0.9712 - val_loss: 0.0617
Epoch 2/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9792 - loss: 0.0466 - val_accuracy: 0.9915 - val_loss: 0.0423
Epoch 3/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9826 - loss: 0.0403 - val_accuracy: 0.9905 - val_loss: 0.0465
Epoch 4/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9876 - loss: 0.0314 - val_accuracy: 0.9902 - val_loss: 0.0398
Epoch 5/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9874 - loss: 0.0317 - val_accuracy: 0.9827 - val_loss: 0.0447
Epoch 6/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9882 - loss: 0.0306 - val_accuracy: 0.9831 - val_loss: 0.0413
Epoch 7/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9899 - loss: 0.0257 - val_accuracy: 0.9836 - val_loss: 0.0397
Epoch 8/15
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9880 - loss: 0.0291 - val

## Step 11: Evaluate on Test Set

The test set contains data the model has never seen during training.

In [21]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"✅ Test accuracy: {test_accuracy*100:.2f}%")
print(f"Test loss:     {test_loss:.4f}\n")

y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

✅ Test accuracy: 99.02%
Test loss:     0.0398

Classification report:
              precision    recall  f1-score   support

      BENIGN       1.00      0.99      1.00     33611
  Web Attack       0.57      0.99      0.72       436

    accuracy                           0.99     34047
   macro avg       0.78      0.99      0.86     34047
weighted avg       0.99      0.99      0.99     34047



## Step 12: Convert to TensorFlow.js Format

This produces:
- `model.json` — the architecture
- `group1-shard1of1.bin` — the trained weights
- `preprocessing.json` — feature names + normalization parameters + class names

All three files go into the PWA at `frontend/public/model/`.

In [22]:
import tensorflowjs as tfjs

# Create output directory
output_dir = 'tfjs_model'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Save the model in TF.js format
tfjs.converters.save_keras_model(model, output_dir)

# Save preprocessing info — the PWA needs this to apply the same normalization
preprocessing = {
    'feature_names': feature_cols,
    'feature_mean': scaler.mean_.tolist(),
    'feature_std': scaler.scale_.tolist(),
    'class_names': class_names,
    'test_accuracy': float(test_accuracy),
    'dataset': 'CIC-IDS-2017 (Thursday web attacks)',
}
with open(f'{output_dir}/preprocessing.json', 'w') as f:
    json.dump(preprocessing, f, indent=2)

print(f"\n✅ Model saved!")
for f in os.listdir(output_dir):
    size = os.path.getsize(f'{output_dir}/{f}') / 1024
    print(f"  📄 {f}  ({size:.2f} KB)")

failed to lookup keras version from the file,
    this is likely a weight only file

✅ Model saved!
  📄 model.json  (4.04 KB)
  📄 preprocessing.json  (5.33 KB)
  📄 group1-shard1of1.bin  (72.26 KB)


## Step 13: Test the Model with Sample Inputs

Verify the trained model produces sensible predictions on a handful of test examples before exporting.

In [23]:
# Pick 5 random samples from the test set
import random
random.seed(42)
sample_indices = random.sample(range(len(X_test)), 5)

print("🧪 Sample predictions:\n")
for i, idx in enumerate(sample_indices):
    sample = X_test[idx:idx+1]
    true_label = class_names[y_test[idx]]
    pred_probs = model.predict(sample, verbose=0)[0]
    pred_label = class_names[pred_probs.argmax()]
    confidence = pred_probs.max()

    status = "✅" if pred_label == true_label else "❌"
    print(f"Sample {i+1}:")
    print(f"  True label:      {true_label}")
    print(f"  Predicted:       {pred_label}  ({confidence*100:.1f}% confidence)")
    print(f"  {status}  {'Correct' if pred_label == true_label else 'Wrong'}\n")

🧪 Sample predictions:

Sample 1:
  True label:      BENIGN
  Predicted:       BENIGN  (100.0% confidence)
  ✅  Correct

Sample 2:
  True label:      BENIGN
  Predicted:       BENIGN  (82.6% confidence)
  ✅  Correct

Sample 3:
  True label:      BENIGN
  Predicted:       BENIGN  (100.0% confidence)
  ✅  Correct

Sample 4:
  True label:      BENIGN
  Predicted:       BENIGN  (100.0% confidence)
  ✅  Correct

Sample 5:
  True label:      BENIGN
  Predicted:       Web Attack  (53.7% confidence)
  ❌  Wrong



## Step 14: Download the Model

Zip the output folder and download it. Then unzip and copy the three files into `frontend/public/model/` in your PWA project.

In [24]:
shutil.make_archive('tfjs_model', 'zip', output_dir)
print(f"\n✅ Done — accuracy {test_accuracy*100:.2f}%")
print(f"\n📦 Download 'tfjs_model.zip' from the file browser on the left.")
print(f"\nFiles inside the zip:")
print(f"  • model.json — the neural network architecture")
print(f"  • group1-shard1of1.bin — the trained weights")
print(f"  • preprocessing.json — feature names and normalization parameters")


✅ Done — accuracy 99.02%

📦 Download 'tfjs_model.zip' from the file browser on the left.

Files inside the zip:
  • model.json — the neural network architecture
  • group1-shard1of1.bin — the trained weights
  • preprocessing.json — feature names and normalization parameters
